In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib

# ===============================
# 1. Load Data
# ===============================
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

TARGET_COL = "label"   # change if needed

X = train_df.drop(columns=[TARGET_COL])
y = train_df[TARGET_COL]

# ===============================
# 2. Handle Missing Values
# ===============================
X = X.replace(-999, np.nan)
X = X.fillna(X.mean())

test_df = test_df.replace(-999, np.nan)
test_df = test_df.fillna(test_df.mean())

# ===============================
# 3. Train-Validation Split
# ===============================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ===============================
# 4. Pipelines + Parameter Grids
# ===============================
pipelines = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ]),

    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(random_state=42))
    ]),

    "Gradient Boosting": Pipeline([
        ("scaler", StandardScaler()),
        ("model", GradientBoostingClassifier(random_state=42))
    ])
}

param_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1, 10],
        "model__class_weight": ["balanced"]
    },

    "Random Forest": {
        "model__n_estimators": [200, 300],
        "model__max_depth": [10, 15],
        "model__min_samples_split": [5, 10],
        "model__class_weight": ["balanced"]
    },

    "Gradient Boosting": {
        "model__n_estimators": [200, 300],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [3, 5],
        "model__subsample": [0.8, 1.0]
    }
}

# ===============================
# 5. Grid Search Training
# ===============================
best_models = {}
results = {}

for name in pipelines:
    print(f"\nRunning Grid Search for {name}...")

    grid = GridSearchCV(
        pipelines[name],
        param_grids[name],
        scoring="roc_auc",
        cv=3,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_

    y_pred = grid.predict(X_val)
    y_prob = grid.predict_proba(X_val)[:, 1]

    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    roc = roc_auc_score(y_val, y_prob)

    results[name] = roc

    print(f"Best Params: {grid.best_params_}")
    print(f"Accuracy  : {acc:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"ROC-AUC   : {roc:.4f}")

# ===============================
# 6. Select Best Model
# ===============================
best_model_name = max(results, key=results.get)
best_model = best_models[best_model_name]

print("\n===============================")
print(f"Best Model Selected: {best_model_name}")
print("===============================")

# ===============================
# 7. Retrain on Full Training Data
# ===============================
best_model.fit(X, y)

# ===============================
# 8. Save Model
# ===============================
joblib.dump(best_model, "best_model.pkl")
print("Best model saved.")

# ===============================
# 9. Predict on Test Set
# ===============================
test_predictions = best_model.predict(test_df)

submission = pd.DataFrame({
    "Prediction": test_predictions
})

submission.to_csv("subCSV.csv", index=False)
print("subCSV.csv generated successfully.")



Running Grid Search for Logistic Regression...
Best Params: {'model__C': 10, 'model__class_weight': 'balanced'}
Accuracy  : 0.6358
F1 Score  : 0.6584
ROC-AUC   : 0.6793

Running Grid Search for Random Forest...
Best Params: {'model__class_weight': 'balanced', 'model__max_depth': 15, 'model__min_samples_split': 10, 'model__n_estimators': 300}
Accuracy  : 0.7106
F1 Score  : 0.7278
ROC-AUC   : 0.7889

Running Grid Search for Gradient Boosting...
Best Params: {'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__n_estimators': 300, 'model__subsample': 0.8}
Accuracy  : 0.7159
F1 Score  : 0.7359
ROC-AUC   : 0.7965

Best Model Selected: Gradient Boosting
Best model saved.
subCSV.csv generated successfully.
